### 1. **Necessity, sufficiency, causality**

Yesterday Lecturer 1 described five questions to ask about any defense. Today we focus on the third one, **guarantee or signal?**, in a slightly more general sense:

— **To what extent is it possible** to get a **guarantee** that $x$ affected the outcome $y$
- **How to tell a guarantee** from a signal
- **How to get** the most meaningful signal


Let's start from yesterday's email example. Say we have a filter for incoming emails. Empirically we found that the filter catches 99 %, someone tries to get through it 100 times, and at least one attempt gets through with probability 63% ($1-0,99^{100}$).

Let's move this problem into the interpretability setting: suppose the filter here plays not the role of a shield but the role of a way to understand "what affected the answer/action/state"?

Hence the first point:
— A filter for defense is one thing
— A way to analyze "what had an effect" is another.

To answer the question "what had an effect" we need two components:
- **necessity**: had there been no $x$ $\Rightarrow$ there would have been no $y$ ($\neg x \Rightarrow \neg y$);
- **sufficiency**: had there been $x$ $\Rightarrow$ there would have been $y$ ($x \Rightarrow y$).


Rigorous formulations of this idea:
- J. Pearl, *Probabilities of causation: three counterfactual interpretations
  and their identification*, Synthese 121, 1999.
  [r260-reprint.pdf](https://ftp.cs.ucla.edu/pub/stat_ser/r260-reprint.pdf)
- D. Watson, L. Gultchin, A. Taly, L. Floridi, *Local explanations via necessity
  and sufficiency: unifying theory and practice*, UAI 2021.
  [PMLR v161](https://proceedings.mlr.press/v161/watson21a.html)

## Probabilistic meaning
(Pearl 1999, defs. 7–9)

Let $x$ be the cause (the tool was there, the rule fired), $y$ the outcome (the answer was produced, the attack was blocked), $Y_{x'}$ what the outcome would have been had $x$ not been there.

**Probability of necessity.** $x$ was there, $y$ happened. What is the probability that without $x$ the outcome would not have happened:

$$\text{PN} = P(Y_{x'} = y' \mid X = x,\ Y = y)$$

**Probability of sufficiency.** $x$ was not there, $y$ did not happen. What is the probability that with $x$ the outcome would have appeared:

$$\text{PS} = P(Y_{x} = y \mid X = x',\ Y = y')$$

**Necessity and sufficiency together:** $\text{PNS} = P(Y_x = y,\ Y_{x'} = y')$.

**Lemma 1:** $\ \text{PNS} = P(x, y)\cdot\text{PN} + P(x', y')\cdot\text{PS}$

<details>
<summary><b>Proof:</b> law of total probability + consistency</summary>

Split over the four hypotheses about what we observed:

$$
\begin{aligned}
\text{PNS} = P(Y_x = y,\ Y_{x'} = y')
&= P(x, y)\cdot P(Y_x = y,\ Y_{x'} = y' \mid x, y) \\
&+ P(x, y')\cdot P(Y_x = y,\ Y_{x'} = y' \mid x, y') \\
&+ P(x', y)\cdot P(Y_x = y,\ Y_{x'} = y' \mid x', y) \\
&+ P(x', y')\cdot P(Y_x = y,\ Y_{x'} = y' \mid x', y') \\[4pt]
&= P(x, y)\cdot \underbrace{P(Y_{x'} = y' \mid x, y)}_{\text{PN}}
 \;+\; P(x, y')\cdot 0
 \;+\; P(x', y)\cdot 0
 \;+\; P(x', y')\cdot \underbrace{P(Y_{x} = y \mid x', y')}_{\text{PS}}
\end{aligned}
$$

The step in the second line rests on the **consistency axiom**: if
we observed $X = x$, then the counterfactual at that same value coincides with the fact,
$Y_x = Y$. Hence:

- for $(x, y)$ the condition $Y_x = y$ already holds, what remains is
  $P(Y_{x'} = y' \mid x, y)$, which is PN;
- for $(x', y')$ $Y_{x'} = y'$ already holds, what remains is
  $P(Y_x = y \mid x', y')$, which is PS;
- for $(x, y')$ consistency gives $Y_x = y' \ne y$, the event is impossible, zero;
- for $(x', y)$ it gives $Y_{x'} = y \ne y'$, also zero.

PNS is computed only over the cases where the outcome matched the cause: in them
necessity and sufficiency are the same event seen from two sides.

</details>


In [1]:
from itertools import combinations, product
from math import factorial

import numpy as np
import pandas as pd

pd.set_option("display.precision", 3)

## Example: who stopped EchoLeak

**Why this example.** Yesterday (Lecturer 1, slides 10–11, 16–17) EchoLeak could be stopped
in two ways: with a filter for incoming emails ($F$, catches 99 %, a signal) and with a ban
on loading URLs from the answer ($B$, a guarantee). Suppose both are on, and there was no
leak. A natural question for a report or an incident review: **who
stopped it?**

This question runs into the problem of **redundancy** (in the causality
literature, *overdetermination*): the outcome is secured by two causes at once.
Then "who stopped it" splits into two different questions: would each
defense have been enough on its own (sufficiency), and could we have done without each
(necessity). The answers to them diverge. And the answer to the second question
in our specific case is a counterfactual: nobody saw the same email with the defense
turned off. It can be computed only if you know the model of the world.
Here we know it, because we wrote it down ourselves.

**Model.** $X_F, X_B \in \{0, 1\}$: whether the defense is on; we don't know this
in advance, each is on with probability $1/2$. $U$: whether the filter recognized
this particular email, $P(U = 1) = 0.99$. The outcome "no leak":

$$Y = (X_F \land U) \lor X_B$$


In [2]:
def exact_shapley(v, players):
    n = len(players)
    return {i: sum(factorial(k) * factorial(n - k - 1) / factorial(n)
                   * (v(frozenset(S) | {i}) - v(frozenset(S)))
                   for k in range(n) for S in combinations([p for p in players if p != i], k))
            for i in players}

P_ON, P_CATCH = 0.5, 0.99
worlds = {(F, B, U): (P_ON if F else 1 - P_ON) * (P_ON if B else 1 - P_ON) * (P_CATCH if U else 1 - P_CATCH)
          for F, B, U in product([0, 1], repeat=3)}

def outcome(world, **do):
    # Y in world world = (X_F, X_B, U); do fixes the defenses: counterfactual "same world, different defense"
    w = dict(zip("FBU", world)) | do
    return int((w["F"] and w["U"]) or w["B"])

def probabilities_of_causation(c):
    k = "FB".index(c)
    xy = {w: p for w, p in worlds.items() if w[k] == 1 and outcome(w) == 1}      # defense was on, no leak
    x_y_ = {w: p for w, p in worlds.items() if w[k] == 0 and outcome(w) == 0}    # defense was off, leak
    pn = sum(p for w, p in xy.items() if outcome(w, **{c: 0}) == 0) / sum(xy.values())
    ps = sum(p for w, p in x_y_.items() if outcome(w, **{c: 1}) == 1) / sum(x_y_.values())
    pns = sum(p for w, p in worlds.items() if outcome(w, **{c: 1}) == 1 and outcome(w, **{c: 0}) == 0)
    assert abs(pns - (sum(xy.values()) * pn + sum(x_y_.values()) * ps)) < 1e-12   # Lemma 1
    return pn, ps, pns

def pn_in_our_world(c):
    # our case: both defenses on, no leak; only U is unknown
    ours = {w: p for w, p in worlds.items() if w[0] == 1 and w[1] == 1}
    return sum(p for w, p in ours.items() if outcome(w, **{c: 0}) == 0) / sum(ours.values())

def v(S):
    # value function as in SHAP: defenses from S are on, the rest from the background
    return sum(p * outcome(w, **{c: 1 for c in S}) for w, p in worlds.items())

phi = exact_shapley(v, ["F", "B"])
table = pd.DataFrame({c: (*probabilities_of_causation(c), pn_in_our_world(c), phi[c]) for c in "FB"},
                     index=["PN", "PS", "PNS", "PN in our case", "Shapley"]).T
table.index = ["F: filter (signal)", "B: URL ban (guarantee)"]
table


,PN,PS,PNS,PN in our case,Shapley
F: filter (signal),0.497,0.99,0.495,0.00,0.124
B: URL ban (guarantee),0.505,1.00,0.505,0.01,0.129


### The same calculation by hand

Let $p = P(X_F = 1) = P(X_B = 1) = 1/2$ and $q = P(U = 1) = 0.99$.
All three variables are independent, $Y = (X_F \land U) \lor X_B$.

The counterfactual is "the same world ($X_B$ and $U$ the same), a different defense". First
let's write down what $Y$ equals if the defense is forced on or off:

| | defense on | defense off |
|---|---|---|
| filter $F$ | $Y_{F=1} = U \lor X_B$ | $Y_{F=0} = X_B$ |
| ban $B$ | $Y_{B=1} = 1$ | $Y_{B=0} = X_F \land U$ |

**Filter $F$.**

- $\text{PNS}(F) = P(Y_{F=1} = 1,\ Y_{F=0} = 0) = P(X_B = 0,\ U = 1) = (1-p)\,q = 0.5 \cdot 0.99 = 0.495$.
  The filter is both necessary and sufficient exactly in the worlds where there is no ban and it recognizes the email.
- $\text{PN}(F) = P(Y_{F=0} = 0 \mid X_F = 1, Y = 1)$. With the filter on
  $Y = U \lor X_B$, so
  $$\text{PN}(F) = \frac{P(X_B = 0,\ U = 1)}{P(U \lor X_B)} = \frac{(1-p)\,q}{1 - (1-p)(1-q)} = \frac{0.495}{0.995} = 0.4975.$$
- $\text{PS}(F) = P(Y_{F=1} = 1 \mid X_F = 0, Y = 0)$. Without the filter $Y = 0$
  means $X_B = 0$, and then $Y_{F=1} = U$: $\ \text{PS}(F) = q = 0.99$.

**Ban $B$.**

- $\text{PNS}(B) = P(Y_{B=1} = 1,\ Y_{B=0} = 0) = P(X_F \land U = 0) = 1 - p\,q = 0.505$.
- $\text{PN}(B) = P(Y_{B=0} = 0 \mid X_B = 1, Y = 1)$. With the ban on, $Y = 1$
  always, so the condition narrows nothing: $\ \text{PN}(B) = P(X_F \land U = 0) = 0.505$.
- $\text{PS}(B) = P(Y_{B=1} = 1 \mid X_B = 0, Y = 0) = 1$: the ban holds for any model answer.

**Check with Lemma 1** $\ \text{PNS} = P(x, y)\,\text{PN} + P(x', y')\,\text{PS}$:

- for $F$: $P(X_F{=}1, Y{=}1) = 0.5 \cdot 0.995 = 0.4975$, $\ P(X_F{=}0, Y{=}0) = 0.5 \cdot 0.5 = 0.25$;
  $\ 0.4975 \cdot 0.4975 + 0.25 \cdot 0.99 = 0.2475 + 0.2475 = 0.495$ ✓
- for $B$: $P(X_B{=}1, Y{=}1) = 0.5$, $\ P(X_B{=}0, Y{=}0) = 0.5 \cdot 0.505 = 0.2525$;
  $\ 0.5 \cdot 0.505 + 0.2525 \cdot 1 = 0.505$ ✓

**Our case: both defenses on, no leak.** Here $X_F = X_B = 1$ is known,
only $U$ is unknown.

- $\text{PN}(F) = P(Y_{F=0} = 0 \mid X_F = X_B = 1) = P(X_B = 0 \mid X_B = 1) = 0$:
  the ban holds even without the filter.
- $\text{PN}(B) = P(Y_{B=0} = 0 \mid X_F = X_B = 1) = P(U = 0) = 1 - q = 0.01$:
  without the ban a leak happens only with an email that the filter misses.

PN "over the population" is about 0.5 because it averages over configurations
where the other defense is off. In our specific case it is on,
and the necessity of each defense drops almost to zero.

### Shapley by hand: where "almost equal" comes from

Value function as in SHAP: $v(S)$ is the probability that there is no leak if the defenses
from $S$ are forced on and the rest are taken from the background (on with probability $p$).

| coalition $S$ | $Y$ under this fixing | $v(S)$ |
|---|---|---|
| $\varnothing$ | $(X_F \land U) \lor X_B$ | $p + (1-p)\,p\,q = 0.7475$ |
| $\{F\}$ | $U \lor X_B$ | $1 - (1-p)(1-q) = 0.995$ |
| $\{B\}$ | $1$ | $1$ |
| $\{F, B\}$ | $1$ | $1$ |

There are two players, two orders of addition, each with weight $\tfrac12$:

$$\varphi_F = \tfrac12\big[\underbrace{v(\{F\}) - v(\varnothing)}_{(1-p)^2 q\ =\ 0.2475} + \underbrace{v(\{F,B\}) - v(\{B\})}_{0}\big] = 0.124$$

$$\varphi_B = \tfrac12\big[\underbrace{v(\{B\}) - v(\varnothing)}_{(1-p)(1-pq)\ =\ 0.2525} + \underbrace{v(\{F,B\}) - v(\{F\})}_{(1-p)(1-q)\ =\ 0.005}\big] = 0.129$$

Efficiency check: $\varphi_F + \varphi_B = 0.2525 = v(\{F,B\}) - v(\varnothing)$ ✓

- **First**, each defense adds almost the same amount, 0.2475 and 0.2525:
  each one is enough to close the leak. This is sufficiency.
- **Second**, it adds almost nothing: the outcome is already secured.
  The filter after the ban adds exactly 0, the ban after the filter adds 0.005, the emails
  that the filter misses.
- The whole "guarantee vs. signal" difference comes down to
  $\varphi_B - \varphi_F = (1-p)(1-q) = 0.005$.

Shapley asks "how much will the defense add if we turn it on", and on average
over orders both add about half of the total gain. The question
"could we have done without it" never appears in this sum.

What we see here:

- **Sufficiency** of both defenses is almost one: each on its own
  stops the leak (the filter, in 99 % of emails).
- **Necessity in our case**, when both are on: zero for the filter
  (the ban holds without it), 0.01 for the ban (without it a leak would happen
  only in the 1 % of emails that the filter missed). Neither defense
  is necessary on its own.
- **Shapley** splits the contribution almost equally, 0.124 and 0.129, the same way
  sufficiency is split almost equally. The answer "both are about equally
  important" is not a computation error. It is the answer to a **different question**.

This is the **redundancy theorem**: if $i$ and $j$ are interchangeable, symmetry
and efficiency force $\varphi_i = \varphi_j > 0$, even though
$\text{PN}(i) = \text{PN}(j) = 0$. With $k$ interchangeable components
each gets $1/k$ of the total contribution: the more the detector rules overlap,
the more negligible each one looks. Which question
Shapley answers is explained by Watson et al.


## What the SHAP value function measures

Watson et al. (2021) define, for a factor $c$ and an outcome $y$ on a distribution $D$
(defs. 2 and 3):

$$\text{PS}(c, y) = P_D\big(f(z) = y \mid c(z) = 1\big), \qquad
  \text{PN}(c, y) = P_D\big(c(z) = 1 \mid f(z) = y\big)$$

These are the precision and recall of an imaginary classifier "$c$ predicts $y$".
They are not Pearl's counterfactual PN and PS, but exactly the same pair of questions:
"is $c$ enough" and "does $y$ do without $c$".

**Proposition 1.** Let $c_S(z) = 1$ if $z$ is built by fixing $x_S$
and sampling the remaining features from $D(\cdot \mid S)$. Then
$v(S) = \text{PS}(c_S, y)$.

That is, the Shapley value is the average gain in **sufficiency** of a random
subset when a feature is added. The question "without what would the answer have been different"
is not asked in this construction. 

## A typical ML case: irises

A flower classifier on four features. We train on 70 % of the sample
and explain flowers from the held-out 30 %; the background $D$ for explanations is the training part.
We compare three numbers per feature:

- **Shapley** with $v(S) = P_D\big(f(x_S, X_R) = \hat y\big)$. Exact, 16 coalitions.
  By Proposition 1 this is sufficiency.
- **Necessity of a feature:** replace the feature with values from the training
  sample, keep the rest as in the flower. What share of replacements changes
  the prediction.
- **Necessity of the pair of petal features**: the same, replacing both at once.


In [3]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

iris = load_iris()
names = ["sepal length", "sepal width", "petal length", "petal width"]
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.3,
                                                    stratify=iris.target, random_state=0)
model = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_train, y_train)
print(f"test accuracy: {model.score(X_test, y_test):.3f}   ({len(X_test)} flowers)\n")

def value(x, S, y_hat):
    Z = X_train.copy()
    Z[:, list(S)] = x[list(S)]
    return (model.predict(Z) == y_hat).mean()

def necessity(x, idx, y_hat):
    Z = np.repeat(x[None], len(X_train), axis=0)
    Z[:, idx] = X_train[:, idx]
    return (model.predict(Z) != y_hat).mean()

def explain(x):
    y_hat = model.predict(x[None])[0]
    phi = exact_shapley(lambda S: value(x, S, y_hat), list(range(4)))
    return y_hat, pd.DataFrame({"Shapley": [phi[i] for i in range(4)],
                                "feature PN": [necessity(x, [i], y_hat) for i in range(4)]}, index=names)

for n in (2, 36):
    y_hat, table = explain(X_test[n])
    print(f"test flower #{n}, prediction: {iris.target_names[y_hat]}, "
          f"PN of the petal feature pair = {necessity(X_test[n], [2, 3], y_hat):.3f}")
    print(table.round(3).to_string(), "\n")


test accuracy: 0.978   (45 flowers)



test flower #2, prediction: setosa, PN of the petal feature pair = 0.667
              Shapley  feature PN
sepal length    0.083         0.0
sepal width     0.051         0.0
petal length    0.203         0.0
petal width     0.330         0.0 



test flower #36, prediction: versicolor, PN of the petal feature pair = 0.657
              Shapley  feature PN
sepal length    0.160         0.0
sepal width     0.047         0.0
petal length    0.198         0.0
petal width     0.263         0.0 



In [4]:
# the same over the whole test set: (flower, feature) with a noticeable contribution but not necessary
rows = []
for x in X_test:
    y_hat, table = explain(x)
    for name, r in table.iterrows():
        rows.append({"feature": name, "Shapley": r["Shapley"], "PN": r["feature PN"]})
    rows.append({"feature": "petal pair", "Shapley": np.nan, "PN": necessity(x, [2, 3], y_hat)})
pairs = pd.DataFrame(rows)

single = pairs[pairs["feature"] != "petal pair"]
big = single[single["Shapley"] > 0.05]
print(f"features with Shapley > 0.05: {len(big)} of {len(single)}")
print(f"of them with PN = 0 (feature replaced, the answer never changed): {(big['PN'] == 0).mean():.1%}\n")
print(pairs.groupby("feature", sort=False)["PN"].agg(["mean", lambda s: (s == 0).mean()])
      .set_axis(["mean PN", "share of flowers with PN = 0"], axis=1).round(3).to_string())


features with Shapley > 0.05: 133 of 180
of them with PN = 0 (feature replaced, the answer never changed): 63.2%

              mean PN  share of flowers with PN = 0
feature                                            
sepal length    0.022                         0.956
sepal width     0.018                         0.978
petal length    0.133                         0.622
petal width     0.279                         0.333
petal pair      0.666                         0.000


Flower #2 (setosa) is the whole redundancy theorem on held-out data.
The petal features get the main contribution, and the necessity of each is
**zero**: petal length and petal width duplicate each other, and the forest recognizes setosa
by either of them. The necessity of the pair is two thirds. The sepal features get
a positive Shapley value with zero necessity.

Flower #36 (versicolor): all four features have a positive contribution,
and not one of them is necessary.

Over the whole test set this is not an exception: 63 % of features with a noticeable contribution
(Shapley > 0.05) have zero necessity, while the pair of petal features
does not have zero necessity on a single flower, the mean PN of the pair is 0.67.

If the question were "what can be dropped from the features without changing the answer",
Shapley would answer "nothing, everything matters". The right answer is "any one of them".


## What we take into Act 2

1. **PN and PS are different quantities.** In EchoLeak both defenses have PS of almost 1,
   and PN in our case of almost 0.
2. **SHAP-like methods measure sufficiency** (Watson, Prop. 1).
   They say nothing about necessity by construction, not because of the implementation.
3. **Under redundancy they systematically err toward "everything matters"**
   (EchoLeak, irises).
4. **Necessity in a specific case is a counterfactual.** It can be computed
   only if you know the model of the world. An agent has no such model: necessity
   will have to be measured by intervention (Act 1) or recorded along the way (Act 2).

Next, the same thing, but for an agent: its components are tools and steps,
each coalition is a separate model run, and the outcome is stochastic.